# Polymer Property Prediction — Round 2

Predicts seven polymer properties from PSMILES. Metric is the **unweighted mean R² across the
seven targets**, which drives every design choice here.

Five of the seven properties (`egb`, `eps`, `nc`, `ei`, `eea`) have only ~220–340 training
labels each, yet contribute **5/7 of the score**. Everything below targets those.

### Pipeline
LightGBM · XGBoost · CatBoost · multi-task NN (5-seed averaged) · SMILES 1D-CNN → Ridge stacking
→ physics blending → range clipping.

### Two ideas doing most of the work

**1. TRUE co-observed partner features.** The six DFT properties are co-observed in *both* train
and test at matching rates — for `eps` rows, `nc` is known for ~59% of train and ~62% of test
molecules. A partner's true value is a legitimate feature: it is a different target, equally
available at inference. Requires SMILES canonicalisation first (10,605 raw strings are only
8,990 distinct molecules).

**2. Physics blending.** Measured as *direct, unfitted* estimators on co-observed molecules:

```
ei  ~= egc + eea     R2 = 0.963      (fundamental gap)
eea ~= ei  - egc     R2 = 0.971
egb ~= egc           R2 = 0.892
eps ~= nc^2          R2 = 0.843      (Maxwell)
```

A tree splits one axis at a time and cannot represent a sum of two columns, so as one feature
among ~5400 these get badly under-weighted. On covered rows the physics estimate reaches ~0.96
where the stack reaches ~0.89 — so it is applied as an explicit calibrated blend after stacking,
with the weight fitted on train OOF and shrunk 25%.

### Leakage control — read before editing

`true_egc` **is the answer** when the target is `egc`, and `ph_ei = egc + eea` leaks whenever the
target is `egc` or `eea`. Every engineered column declares which labels it is built from
(`USES`), and two different guards apply:

- **per-property models** (LGBM/XGB/CatBoost) → `drop_leaky()` removes the offending columns
- **the multi-task NN**, which trains on all properties at once → `mask_rows_for_multitask()`
  neutralises them per **row**. Column-dropping is wrong there: a column that leaks for `egc`
  rows is legitimate for `eps` rows.

The CNN reads SMILES only and needs no guard. Section 5 proves the leak exists, then proves each
guard removes it, and **aborts** if either check fails.

### Compliance
Uses **only** `train.csv` and `test.csv`. The `archive/` folder is never read — asserted at
runtime in sections 4 and 11. No external data, no pretrained weights. All seeds fixed.

**Runtime ≈ 4.5 h on a T4 GPU.**

In [1]:
!pip install rdkit -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.4/37.4 MB 45.6 MB/s eta 0:00:00


In [2]:
import os, sys, time, pickle, warnings, gc, re, glob
import numpy as np
import pandas as pd
from datetime import datetime
warnings.filterwarnings('ignore')

import lightgbm as lgb
import xgboost as xgb
import catboost as cb
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from rdkit import Chem, RDLogger
from rdkit.Chem import (Descriptors, AllChem, MACCSkeys, rdMolDescriptors,
                        Lipinski, rdFingerprintGenerator)
RDLogger.logger().setLevel(RDLogger.ERROR)

SEED         = 42
N_FOLDS      = 10
NN_SEEDS     = [42, 202, 777, 1337, 2024]   # averaging these is worth ~+0.015 on the NN alone
TARGET_TYPES = ['tg', 'egc', 'egb', 'eps', 'nc', 'ei', 'eea']
DFT_PROPS    = ['egc', 'egb', 'ei', 'eea', 'eps', 'nc']   # co-observed block ('tg' is disjoint)
MORGAN_BITS_R2, MORGAN_BITS_R3, AP_BITS, TT_BITS = 2048, 1024, 1024, 1024

np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

_KAGGLE_PATHS = ['/kaggle/input/competitions/ppp-round-2', '/kaggle/input/ppp-round-2',
                 '/kaggle/input/aisehack-2-0']
DATA_DIR = next((p for p in _KAGGLE_PATHS if os.path.exists(p)), os.getcwd())
WORK_DIR = '/kaggle/working' if os.path.exists('/kaggle/working') else os.getcwd()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class Logger:
    def __init__(self): self.t0 = time.time()
    def _p(self, lv, m):
        print(f'[{datetime.now().strftime("%H:%M:%S")}] [{lv:>6}] {m}', flush=True)
    def info(self, m): self._p('INFO', m)
    def metric(self, m): self._p('METRIC', m)
    def ok(self, m): self._p('OK', m)
    def warn(self, m): self._p('WARN', m)
    def header(self, m):
        self._p('INFO', '=' * 60); self._p('INFO', f'  {m}'); self._p('INFO', '=' * 60)
    def sub(self, m): self._p('INFO', f'--- {m} ---')
log = Logger()

print(f'device={device}  data={DATA_DIR}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

device=cuda  data=/kaggle/input/competitions/ppp-round-2
GPU: Tesla T4


## 1. Hyperparameters

Capacity is **scaled to sample count**. A single parameter set sized for `tg` (4143 rows) badly
over-fits the ~220-row properties: measured on standalone LightGBM, size-adaptive settings gain
**+0.0050** mean R², concentrated exactly where it is needed (`ei` +0.018, `nc` +0.009,
`eps` +0.007) while leaving `tg`/`egc` unchanged.

In [3]:
LGBM_BASE = dict(objective='regression', metric='rmse', boosting_type='gbdt',
                 n_estimators=3000, learning_rate=0.015, max_depth=7, num_leaves=63,
                 min_child_samples=10, reg_alpha=0.1, reg_lambda=1.0,
                 subsample=0.8, colsample_bytree=0.6,
                 random_state=SEED, verbose=-1, n_jobs=-1)

XGB_BASE = dict(objective='reg:squarederror', n_estimators=3000, learning_rate=0.015,
                max_depth=7, subsample=0.8, colsample_bytree=0.6,
                reg_alpha=0.1, reg_lambda=1.0, min_child_weight=10,
                random_state=SEED, verbosity=0)
if torch.cuda.is_available():
    XGB_BASE['device'] = 'cuda'

CB_BASE = dict(iterations=3000, learning_rate=0.03, depth=7, l2_leaf_reg=3.0,
               random_seed=SEED, verbose=0, od_type='Iter', od_wait=100)
if torch.cuda.is_available():
    CB_BASE['task_type'] = 'GPU'; CB_BASE['devices'] = '0'

SMALL = 600          # below this many rows, shrink the model

def lgbm_params(n):
    if n < SMALL:
        return dict(LGBM_BASE, num_leaves=7, max_depth=4, min_child_samples=5,
                    colsample_bytree=0.20, learning_rate=0.02, n_estimators=1500)
    return LGBM_BASE

def xgb_params(n):
    if n < SMALL:
        return dict(XGB_BASE, max_depth=3, min_child_weight=5,
                    colsample_bytree=0.20, learning_rate=0.02)
    return XGB_BASE

def cb_params(n):
    if n < SMALL:
        return dict(CB_BASE, depth=4, learning_rate=0.02)
    return CB_BASE

NN_CFG  = dict(hidden_dims=[1024, 512, 256, 128], head_dim=64, dropout=0.3,
               lr=1e-3, weight_decay=1e-4, epochs=200, batch_size=64, patience=25)
CNN_CFG = dict(embed_dim=64, n_filters=128, kernel_sizes=[3, 5, 7, 11], fc_dim=256,
               dropout=0.3, max_len=200, n_aug=5, lr=5e-4, weight_decay=1e-4,
               epochs=120, batch_size=64, patience=20)
print('hyperparameters set')

hyperparameters set


## 2. Load data

In [4]:
log.header('LOADING DATA')
train_df = pd.read_csv(f'{DATA_DIR}/train.csv')
test_df  = pd.read_csv(f'{DATA_DIR}/test.csv')

train_df = train_df.drop_duplicates(subset=['smiles', 'target_type', 'target'])
# reset_index is REQUIRED: train_features gets a fresh 0..n-1 index, and boolean masks taken
# from train_df align by index. A stale index silently misaligns every model.
train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

log.info(f'train {train_df.shape}  test {test_df.shape}')
for tt in TARGET_TYPES:
    log.info(f'  {tt}: {(train_df.target_type==tt).sum()} train, '
             f'{(test_df.target_type==tt).sum()} test')

[10:25:14] [  INFO] ============================================================
[10:25:14] [  INFO]   LOADING DATA
[10:25:14] [  INFO] ============================================================
[10:25:14] [  INFO] train (7409, 3)  test (4940, 3)
[10:25:14] [  INFO]   tg: 4143 train, 2763 test
[10:25:14] [  INFO]   egc: 2028 train, 1352 test
[10:25:14] [  INFO]   egb: 337 train, 224 test
[10:25:14] [  INFO]   eps: 229 train, 153 test
[10:25:14] [  INFO]   nc: 229 train, 153 test
[10:25:14] [  INFO]   ei: 222 train, 148 test
[10:25:14] [  INFO]   eea: 221 train, 147 test


## 3. Featurization

RDKit descriptors + Morgan(r=2,3) + AtomPair + Topological-Torsion + MACCS + polymer-specific
terms (backbone length between the two `*` connection points, conjugation ratio, Gasteiger
charge statistics) + SMARTS functional-group counts.

In [5]:
GROUP_SMARTS = {
    'aromatic_6': '[a]1[a][a][a][a][a]1', 'aromatic_5': '[a]1[a][a][a][a]1',
    'amide': '[NX3][CX3](=[OX1])', 'ester': '[CX3](=[OX1])[OX2]',
    'ether': '[OD2]([#6])[#6]', 'hydroxyl': '[OX2H]', 'carbonyl': '[CX3]=[OX1]',
    'carboxyl': '[CX3](=[OX1])[OX2H1]', 'sulfonyl': '[#16X4](=[OX1])(=[OX1])',
    'imide': '[CX3](=[OX1])[NX3][CX3](=[OX1])', 'urea': '[NX3][CX3](=[OX1])[NX3]',
    'cyano': '[CX2]#[NX1]', 'nitro': '[NX3+](=O)[O-]', 'fluorine': '[F]',
    'chlorine': '[Cl]', 'bromine': '[Br]', 'silicon': '[Si]', 'phosphorus': '[P]',
    'double_bond': '[CX3]=[CX3]', 'triple_bond': '[CX2]#[CX2]', 'epoxide': 'C1OC1',
    'azo': '[NX2]=[NX2]', 'thioether': '[#16X2]([#6])[#6]', 'amine_primary': '[NX3H2]',
    'amine_secondary': '[NX3H1]([#6])[#6]', 'amine_tertiary': '[NX3]([#6])([#6])[#6]',
    'phenol': '[OX2H][c]', 'vinyl': '[CX3]=[CX3H1]', 'methyl': '[CH3]',
    'trifluoromethyl': '[CX4](F)(F)F', 'anhydride': '[CX3](=[OX1])[OX2][CX3](=[OX1])',
}
GROUP_PATTERNS = {k: p for k, s in GROUP_SMARTS.items()
                  if (p := Chem.MolFromSmarts(s)) is not None}

def compute_custom(mol, smi):
    f = {}
    if mol is None: return f
    try:
        f['n_star'] = smi.count('*'); f['smi_len'] = len(smi)
        f['n_atoms'] = mol.GetNumAtoms(); f['n_heavy'] = mol.GetNumHeavyAtoms()
        f['n_bonds'] = mol.GetNumBonds()
        f['n_rings'] = mol.GetRingInfo().NumRings()
        f['n_arom_rings'] = rdMolDescriptors.CalcNumAromaticRings(mol)
        f['n_aliph_rings'] = rdMolDescriptors.CalcNumAliphaticRings(mol)
        f['arom_ratio'] = f['n_arom_rings'] / max(f['n_rings'], 1)
        f['ring_ratio'] = f['n_rings'] / max(f['n_atoms'], 1)
        f['n_rot'] = Lipinski.NumRotatableBonds(mol)
        f['rot_ratio'] = f['n_rot'] / max(f['n_bonds'], 1)
        f['n_het'] = Lipinski.NumHeteroatoms(mol)
        f['het_ratio'] = f['n_het'] / max(f['n_atoms'], 1)
        f['n_hbd'] = Lipinski.NumHDonors(mol); f['n_hba'] = Lipinski.NumHAcceptors(mol)
        f['fsp3'] = rdMolDescriptors.CalcFractionCSP3(mol)
        cj = sum(1 for b in mol.GetBonds() if b.GetIsConjugated())
        f['n_conj_bonds'] = cj; f['conj_ratio'] = cj / max(f['n_bonds'], 1)
        nums = [a.GetAtomicNum() for a in mol.GetAtoms()]
        for z, nm in [(6,'C'),(7,'N'),(8,'O'),(9,'F'),(16,'S'),(17,'Cl'),(35,'Br'),(14,'Si'),(15,'P')]:
            f[f'n_{nm}'] = nums.count(z); f[f'fr_{nm}'] = nums.count(z)/max(len(nums),1)
        stars = [a.GetIdx() for a in mol.GetAtoms() if a.GetSymbol() == '*']
        if len(stars) == 2:
            try:
                path = Chem.rdmolops.GetShortestPath(mol, stars[0], stars[1])
                f['backbone_len'] = len(path) - 2
                ba = sum(1 for i in path[1:-1] if mol.GetAtomWithIdx(i).GetIsAromatic())
                f['backbone_arom_ratio'] = ba / max(f['backbone_len'], 1)
            except Exception:
                f['backbone_len'] = 0; f['backbone_arom_ratio'] = 0.0
        try:
            AllChem.ComputeGasteigerCharges(mol)
            ch = [a.GetDoubleProp('_GasteigerCharge') for a in mol.GetAtoms()]
            ch = [c for c in ch if np.isfinite(c)]
            if ch:
                f['ch_mean']=np.mean(ch); f['ch_std']=np.std(ch)
                f['ch_min']=np.min(ch);  f['ch_max']=np.max(ch)
                f['ch_range']=f['ch_max']-f['ch_min']
        except Exception: pass
        for nm, fn in [('balaban_j', Descriptors.BalabanJ), ('bertz_ct', Descriptors.BertzCT)]:
            try: f[nm] = fn(mol)
            except Exception: pass
    except Exception: pass
    return {k: (0.0 if v is None or (isinstance(v,float) and not np.isfinite(v)) else v)
            for k, v in f.items()}

def featurize_batch(smiles_list):
    n = len(smiles_list); t0 = time.time(); every = max(1, n//10)
    rd_l, m2_l, m3_l, ap_l, tt_l, mc_l, cu_l, gr_l = [], [], [], [], [], [], [], []
    apg = rdFingerprintGenerator.GetAtomPairGenerator(fpSize=AP_BITS)
    ttg = rdFingerprintGenerator.GetTopologicalTorsionGenerator(fpSize=TT_BITS)
    log.info(f'featurizing {n} molecules...')
    for i, smi in enumerate(smiles_list):
        if (i+1) % every == 0:
            el = time.time()-t0
            log.info(f'  {i+1}/{n} ({100*(i+1)/n:.0f}%) ETA {(n-i-1)/max((i+1)/el,.01):.0f}s')
        mol = Chem.MolFromSmiles(smi)
        try:
            d = Descriptors.CalcMolDescriptors(mol) if mol is not None else {}
            rd_l.append({k: (0.0 if v is None or (isinstance(v,float) and not np.isfinite(v))
                             else float(v)) for k, v in d.items()})
        except Exception:
            rd_l.append({})
        if mol is not None:
            m2_l.append(np.array(AllChem.GetMorganFingerprintAsBitVect(mol,2,nBits=MORGAN_BITS_R2), dtype=np.float32))
            m3_l.append(np.array(AllChem.GetMorganFingerprintAsBitVect(mol,3,nBits=MORGAN_BITS_R3), dtype=np.float32))
            ap_l.append(apg.GetFingerprintAsNumPy(mol).astype(np.float32))
            tt_l.append(ttg.GetFingerprintAsNumPy(mol).astype(np.float32))
            mc_l.append(np.array(MACCSkeys.GenMACCSKeys(mol), dtype=np.float32))
        else:
            m2_l.append(np.zeros(MORGAN_BITS_R2, np.float32)); m3_l.append(np.zeros(MORGAN_BITS_R3, np.float32))
            ap_l.append(np.zeros(AP_BITS, np.float32));        tt_l.append(np.zeros(TT_BITS, np.float32))
            mc_l.append(np.zeros(167, np.float32))
        cu_l.append(compute_custom(mol, smi))
        gr_l.append({f'grp_{k}': len(mol.GetSubstructMatches(p)) if mol is not None else 0
                     for k, p in GROUP_PATTERNS.items()})
    log.info(f'done in {time.time()-t0:.0f}s')
    df_rd = pd.DataFrame(rd_l); df_rd.columns = [f'rd_{c}' for c in df_rd.columns]
    df_cu = pd.DataFrame(cu_l); df_cu.columns = [f'po_{c}' for c in df_cu.columns]
    parts = [df_rd,
             pd.DataFrame(np.array(m2_l), columns=[f'mfp2_{i}' for i in range(MORGAN_BITS_R2)]),
             pd.DataFrame(np.array(m3_l), columns=[f'mfp3_{i}' for i in range(MORGAN_BITS_R3)]),
             pd.DataFrame(np.array(ap_l), columns=[f'ap_{i}' for i in range(AP_BITS)]),
             pd.DataFrame(np.array(tt_l), columns=[f'tt_{i}' for i in range(TT_BITS)]),
             pd.DataFrame(np.array(mc_l), columns=[f'mac_{i}' for i in range(167)]),
             df_cu, pd.DataFrame(gr_l)]
    return pd.concat(parts, axis=1)

def clean_features(df):
    df = df.copy()
    fm = float(np.finfo(np.float32).max)
    num = df.select_dtypes(include=[np.number]).columns
    df[num] = df[num].clip(lower=-fm, upper=fm)
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.fillna(0.0, inplace=True)
    obj = df.select_dtypes(include=['object']).columns.tolist()
    if obj: df.drop(columns=obj, inplace=True)
    df.columns = [re.sub(r'[\[\]<,]', '_', c) for c in df.columns]
    return df

log.header('FEATURE ENGINEERING')
CACHE = os.path.join(WORK_DIR, 'features_final.pkl')
if os.path.exists(CACHE):
    train_features, test_features = pickle.load(open(CACHE, 'rb'))
    log.ok(f'loaded cached features {train_features.shape}')
else:
    all_smi = pd.concat([train_df[['smiles']], test_df[['smiles']]]).drop_duplicates('smiles')
    feats = clean_features(featurize_batch(all_smi['smiles'].tolist()))
    feats.index = all_smi['smiles'].values
    train_features = feats.loc[train_df.smiles.values].reset_index(drop=True)
    test_features  = feats.loc[test_df.smiles.values].reset_index(drop=True)
    const = train_features.columns[train_features.nunique() <= 1].tolist()
    if const:
        train_features.drop(columns=const, inplace=True)
        test_features.drop(columns=[c for c in const if c in test_features.columns], inplace=True)
        log.info(f'dropped {len(const)} constant columns')
    del feats; gc.collect()
    pickle.dump((train_features, test_features), open(CACHE, 'wb'), protocol=4)
log.info(f'train_features {train_features.shape}  test_features {test_features.shape}')

[10:25:14] [  INFO] ============================================================
[10:25:14] [  INFO]   FEATURE ENGINEERING
[10:25:14] [  INFO] ============================================================
[10:25:14] [  INFO] featurizing 10605 molecules...
[10:25:40] [  INFO]   1060/10605 (10%) ETA 234s
[10:26:06] [  INFO]   2120/10605 (20%) ETA 209s
[10:26:34] [  INFO]   3180/10605 (30%) ETA 187s
[10:27:03] [  INFO]   4240/10605 (40%) ETA 164s
[10:27:31] [  INFO]   5300/10605 (50%) ETA 138s
[10:28:01] [  INFO]   6360/10605 (60%) ETA 111s
[10:28:30] [  INFO]   7420/10605 (70%) ETA 84s
[10:28:59] [  INFO]   8480/10605 (80%) ETA 57s
[10:29:28] [  INFO]   9540/10605 (90%) ETA 28s
[10:29:56] [  INFO]   10600/10605 (100%) ETA 0s
[10:29:56] [  INFO] done in 282s
[10:30:02] [  INFO] dropped 183 constant columns
[10:30:02] [  INFO] train_features (7409, 5398)  test_features (4940, 5398)


## 4. TRUE co-observed partner features + physics

Built from **`train.csv` only**. A runtime assertion at the end of this cell fails if the
partner table ever grows beyond Round-2 train — that is the archive guard.

In [6]:
log.header('TRUE PARTNER FEATURES')
USES  = {}         # engineered column -> set of properties whose LABEL it uses
FILLS = {}         # column -> fill value computed on TRAIN (applied to train and test alike)

def _canon(s):
    m = Chem.MolFromSmiles(s)
    return Chem.MolToSmiles(m) if m is not None else s

_cmap = {s: _canon(s) for s in set(train_df.smiles) | set(test_df.smiles)}
_tc = train_df.smiles.map(_cmap)
_ec = test_df.smiles.map(_cmap)
log.info(f'canonical molecules: {len(set(_cmap.values()))} of {len(_cmap)} raw SMILES')

_tmp = train_df.assign(_c=_tc)
_truth = {q: _tmp[_tmp.target_type == q].groupby('_c').target.mean() for q in DFT_PROPS}
log.info('partner table: ' + ', '.join(f'{q}={len(_truth[q])}' for q in DFT_PROPS))

raw_tr, raw_te = {}, {}
for q in DFT_PROPS:
    raw_tr[q] = _tc.map(_truth[q]).values.astype(np.float64)
    raw_te[q] = _ec.map(_truth[q]).values.astype(np.float64)

def _add(name, a, b, srcs):
    """Mean-fill (train mean, both sides) + availability flag, so the shared matrix stays
    NaN-free for CatBoost/NN as well as LightGBM/XGBoost."""
    fill = float(np.nanmean(np.where(np.isfinite(a), a, np.nan)))
    FILLS[name] = fill
    train_features[name] = np.where(np.isfinite(a), a, fill)
    test_features[name]  = np.where(np.isfinite(b), b, fill)
    train_features[f'{name}_ok'] = np.isfinite(a).astype(np.float32)
    test_features[f'{name}_ok']  = np.isfinite(b).astype(np.float32)
    FILLS[f'{name}_ok'] = 0.0
    USES[name] = set(srcs); USES[f'{name}_ok'] = set(srcs)

for q in DFT_PROPS:
    _add(f'true_{q}', raw_tr[q], raw_te[q], [q])

# Physics as DIRECT (unfitted) estimators on co-observed molecules.
_add('ph_ei',  raw_tr['egc']+raw_tr['eea'], raw_te['egc']+raw_te['eea'], ['egc','eea'])  # R2=.963
_add('ph_eea', raw_tr['ei']-raw_tr['egc'],  raw_te['ei']-raw_te['egc'],  ['ei','egc'])   # R2=.971
_add('ph_egb', raw_tr['egc'],               raw_te['egc'],               ['egc'])        # R2=.892
_add('ph_eps', raw_tr['nc']**2,             raw_te['nc']**2,             ['nc'])         # Maxwell
_add('ph_nc',  np.sqrt(np.clip(raw_tr['eps'],0,None)),
               np.sqrt(np.clip(raw_te['eps'],0,None)),                   ['eps'])
_add('ph_gap', raw_tr['egb']-raw_tr['egc'], raw_te['egb']-raw_te['egc'], ['egb','egc'])

def drop_leaky(feat_df, target_type):
    """PER-PROPERTY models: remove every column built from this target's label."""
    bad = [c for c, s in USES.items() if target_type in s and c in feat_df.columns]
    return feat_df.drop(columns=bad)

def mask_rows_for_multitask(feat_df, target_types):
    """Multi-task NN: neutralise own-target columns per ROW, using the TRAIN fill value so
    train and test match exactly. Column-dropping is wrong here -- a column that leaks for egc
    rows is legitimate for eps rows."""
    out = feat_df.copy()
    tt = np.asarray(target_types)
    for c, s in USES.items():
        if c in out.columns:
            m = np.isin(tt, list(s))
            if m.any():
                out.loc[m, c] = FILLS[c]
    return out

for q in DFT_PROPS:
    assert len(_truth[q]) == _tmp[_tmp.target_type == q]._c.nunique(), \
        f'{q}: partner table exceeds train.csv -- external labels merged in'
log.ok('COMPLIANCE: partner table built from train.csv only')
log.ok(f'added {len(USES)} engineered columns -> {train_features.shape[1]} features total')

[10:30:02] [  INFO] ============================================================
[10:30:02] [  INFO]   TRUE PARTNER FEATURES
[10:30:02] [  INFO] ============================================================
[10:30:07] [  INFO] canonical molecules: 8990 of 10605 raw SMILES
[10:30:07] [  INFO] partner table: egc=2028, egb=337, ei=222, eea=221, eps=229, nc=229
[10:30:07] [    OK] COMPLIANCE: partner table built from train.csv only
[10:30:07] [    OK] added 24 engineered columns -> 5422 features total


## 5. Leakage assertions

Proves the leak is real first (`true_eps` reproduces the `eps` target exactly), then proves each
guard removes it. A guard that passes without a demonstrable leak proves nothing. **Execution
stops if any check fails.**

In [7]:
log.header('LEAKAGE SELF-TEST')
fail = []

for p in DFT_PROPS:                                     # (a) the leak genuinely exists
    m = (train_df.target_type == p).values
    if not np.allclose(train_features.loc[m, f'true_{p}'], train_df.loc[m, 'target'], atol=1e-6):
        fail.append(f'{p}: true_{p} does NOT reproduce target -- feature build is wrong')
log.info('(a) leak reproduced for all DFT properties (as expected)')

for p in TARGET_TYPES:                                  # (b) drop_leaky removes dependents
    kept = drop_leaky(train_features, p)
    bad = [c for c in kept.columns if p in USES.get(c, set())]
    if bad: fail.append(f'{p}: drop_leaky left {bad}')
log.info('(b) drop_leaky leaves no dependent column')

_mm = mask_rows_for_multitask(train_features, train_df.target_type.values)
for p in DFT_PROPS:                                     # (c) row-mask neutralises for the NN
    m = (train_df.target_type == p).values
    if np.allclose(_mm.loc[m, f'true_{p}'], train_df.loc[m, 'target'], atol=1e-6):
        fail.append(f'{p}: NN row-mask did not neutralise true_{p}')
log.info('(c) NN row-mask neutralises own-target columns')

log.info('(d) partner availability, train vs test (must be close or CV will not transfer):')
for p in DFT_PROPS:                                     # (d) availability match
    mtr = (train_df.target_type == p).values
    mte = (test_df.target_type == p).values
    cols = [f'true_{q}_ok' for q in DFT_PROPS if q != p]
    a = train_features.loc[mtr, cols].sum(1).mean()
    b = test_features.loc[mte,  cols].sum(1).mean()
    flag = '  <-- CHECK' if abs(a - b) > 0.5 else ''
    log.info(f'    {p:4s} mean partner count  train {a:.2f}  test {b:.2f}{flag}')

assert not fail, 'LEAKAGE CHECK FAILED:\n' + '\n'.join(fail)
log.ok('all leakage checks passed')

[10:30:07] [  INFO] ============================================================
[10:30:07] [  INFO]   LEAKAGE SELF-TEST
[10:30:07] [  INFO] ============================================================
[10:30:07] [  INFO] (a) leak reproduced for all DFT properties (as expected)
[10:30:07] [  INFO] (b) drop_leaky leaves no dependent column
[10:30:07] [  INFO] (c) NN row-mask neutralises own-target columns
[10:30:07] [  INFO] (d) partner availability, train vs test (must be close or CV will not transfer):
[10:30:07] [  INFO]     egc  mean partner count  train 0.32  test 0.40
[10:30:07] [  INFO]     egb  mean partner count  train 2.04  test 2.00
[10:30:07] [  INFO]     ei   mean partner count  train 2.76  test 3.01
[10:30:07] [  INFO]     eea  mean partner count  train 2.85  test 2.88
[10:30:07] [  INFO]     eps  mean partner count  train 2.83  test 2.84
[10:30:07] [  INFO]     nc   mean partner count  train 2.83  test 2.84
[10:30:07] [    OK] all leakage checks passed


## 6. LightGBM / XGBoost / CatBoost (per property, size-adaptive)

In [8]:
log.header('GRADIENT BOOSTING')

def cv_tree(kind, X, y, tt):
    """Per-property CV. drop_leaky() is applied by the caller; params scale with len(y)."""
    n = len(y)
    kf = KFold(N_FOLDS, shuffle=True, random_state=SEED)
    oof = np.zeros(n); models = []
    for f, (a, b) in enumerate(kf.split(X)):
        Xa, Xb = X.iloc[a], X.iloc[b]
        ya, yb = y.iloc[a], y.iloc[b]
        if kind == 'lgbm':
            m = lgb.LGBMRegressor(**lgbm_params(n))
            m.fit(Xa, ya, eval_set=[(Xb, yb)],
                  callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)])
        elif kind == 'xgb':
            m = xgb.XGBRegressor(early_stopping_rounds=100, **xgb_params(n))
            m.fit(Xa, ya, eval_set=[(Xb, yb)], verbose=False)
        else:
            m = cb.CatBoostRegressor(**cb_params(n))
            m.fit(Xa.values, ya.values, eval_set=(Xb.values, yb.values), verbose=0)
        oof[b] = m.predict(Xb.values if kind == 'cb' else Xb)
        models.append(m)
    r2 = r2_score(y, oof)
    log.metric(f'  [{tt}] {kind} OOF R2={r2:.4f}  (n={n}, {"small" if n < SMALL else "large"} params)')
    return models, oof, r2

tree_models, tree_oof, tree_r2 = {}, {}, {}
for kind in ['lgbm', 'xgb', 'cb']:
    log.sub(kind)
    tree_models[kind] = {}; tree_r2[kind] = {}
    oof_all = np.zeros(len(train_df))
    for tt in TARGET_TYPES:
        mask = (train_df.target_type == tt).values
        X = drop_leaky(train_features[mask].reset_index(drop=True), tt)
        y = train_df.loc[mask, 'target'].reset_index(drop=True)
        mods, oof, r2 = cv_tree(kind, X, y, tt)
        tree_models[kind][tt] = mods; tree_r2[kind][tt] = r2
        oof_all[mask] = oof
    tree_oof[kind] = oof_all
    log.metric(f'>>> {kind} mean OOF R2 = {np.mean(list(tree_r2[kind].values())):.4f}')

[10:30:07] [  INFO] ============================================================
[10:30:07] [  INFO]   GRADIENT BOOSTING
[10:30:07] [  INFO] ============================================================
[10:30:07] [  INFO] --- lgbm ---
[10:37:44] [METRIC]   [tg] lgbm OOF R2=0.9033  (n=4143, large params)
[10:40:28] [METRIC]   [egc] lgbm OOF R2=0.9084  (n=2028, large params)
[10:40:42] [METRIC]   [egb] lgbm OOF R2=0.9121  (n=337, small params)
[10:40:51] [METRIC]   [eps] lgbm OOF R2=0.7965  (n=229, small params)
[10:41:00] [METRIC]   [nc] lgbm OOF R2=0.8651  (n=229, small params)
[10:41:09] [METRIC]   [ei] lgbm OOF R2=0.8270  (n=222, small params)
[10:41:22] [METRIC]   [eea] lgbm OOF R2=0.8881  (n=221, small params)
[10:41:22] [METRIC] >>> lgbm mean OOF R2 = 0.8715
[10:41:22] [  INFO] --- xgb ---
[10:55:32] [METRIC]   [tg] xgb OOF R2=0.9078  (n=4143, large params)
[11:03:54] [METRIC]   [egc] xgb OOF R2=0.9108  (n=2028, large params)
[11:06:34] [METRIC]   [egb] xgb OOF R2=0.9103  (n=337, 

## 7. Multi-task neural network (5-seed averaged)

The NN is the strongest single model on `eps`, `nc` and `ei` — the three weakest properties —
and a 6M-parameter net fitting ~220-row heads has large seed variance. Averaging five seeds is
worth **~+0.015** on the NN's own OOF. The fold split is held fixed at `random_state=SEED` so
every seed predicts the same held-out rows, which is what makes averaging the OOF valid.

`drop_last=True` is required: at 10 folds one split leaves a final batch of exactly 1 sample,
and `BatchNorm1d` cannot compute a variance from one value.

In [9]:
log.header('MULTI-TASK NN')
task_map = {t: i for i, t in enumerate(TARGET_TYPES)}

class MultiTaskNet(nn.Module):
    def __init__(s, d_in, hidden=(1024,512,256,128), head=64, n_tasks=7, dropout=0.3):
        super().__init__()
        L, prev = [], d_in
        for i, h in enumerate(hidden):
            L += [nn.Linear(prev,h), nn.BatchNorm1d(h), nn.SiLU(),
                  nn.Dropout(max(dropout*(1-i*0.1), 0.05))]
            prev = h
        s.trunk = nn.Sequential(*L)
        s.heads = nn.ModuleList([nn.Sequential(nn.Linear(prev,head), nn.SiLU(),
                                               nn.Dropout(dropout*0.3), nn.Linear(head,1))
                                 for _ in range(n_tasks)])
    def forward(s, x, t):
        z = s.trunk(x)
        out = torch.zeros(x.size(0), device=x.device)
        for i, h in enumerate(s.heads):
            m = (t == i)
            if m.any(): out[m] = h(z[m]).squeeze(-1)
        return out

class DS(Dataset):
    def __init__(s, X, y, t):
        s.X = torch.FloatTensor(X); s.y = torch.FloatTensor(y); s.t = torch.LongTensor(t)
    def __len__(s): return len(s.X)
    def __getitem__(s, i): return s.X[i], s.y[i], s.t[i]

# CRITICAL: neutralise own-target columns per row before the NN ever sees them
X_nn_train = mask_rows_for_multitask(train_features, train_df.target_type.values).values
X_nn_test  = mask_rows_for_multitask(test_features,  test_df.target_type.values).values
X_nn_train = np.nan_to_num(np.clip(X_nn_train, -3.4e38, 3.4e38)).astype(np.float32)
X_nn_test  = np.nan_to_num(np.clip(X_nn_test,  -3.4e38, 3.4e38)).astype(np.float32)
log.ok('NN inputs row-masked')

y_all  = train_df.target.values.astype(np.float32)
t_all  = train_df.target_type.map(task_map).values.astype(np.int64)
t_test = test_df.target_type.map(task_map).values.astype(np.int64)

nn_oof = np.zeros(len(y_all)); nn_fold_models = []

for _si, _sd in enumerate(NN_SEEDS):
  _oof_seed = np.zeros(len(y_all))
  for fold, (tr_i, va_i) in enumerate(KFold(N_FOLDS, shuffle=True, random_state=SEED).split(X_nn_train)):
    t0 = time.time(); torch.manual_seed(_sd + fold)
    sc = StandardScaler()
    Xa = np.nan_to_num(sc.fit_transform(X_nn_train[tr_i])).astype(np.float32)
    Xb = np.nan_to_num(sc.transform(X_nn_train[va_i])).astype(np.float32)
    tsc, ya, yb = {}, y_all[tr_i].copy(), y_all[va_i].copy()
    for tt, i in task_map.items():
        ma, mb = (t_all[tr_i] == i), (t_all[va_i] == i)
        s = StandardScaler()
        if ma.any(): ya[ma] = s.fit_transform(y_all[tr_i][ma].reshape(-1,1)).ravel()
        if mb.any(): yb[mb] = s.transform(y_all[va_i][mb].reshape(-1,1)).ravel()
        tsc[i] = s
    model = MultiTaskNet(Xa.shape[1], NN_CFG['hidden_dims'], NN_CFG['head_dim'],
                         dropout=NN_CFG['dropout']).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=NN_CFG['lr'], weight_decay=NN_CFG['weight_decay'])
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=NN_CFG['epochs'], eta_min=1e-6)
    dl  = DataLoader(DS(Xa, ya, t_all[tr_i]), batch_size=NN_CFG['batch_size'],
                     shuffle=True, drop_last=True)
    vdl = DataLoader(DS(Xb, yb, t_all[va_i]), batch_size=NN_CFG['batch_size']*4)
    best, best_state, pat = 1e18, None, 0
    for ep in range(NN_CFG['epochs']):
        model.train()
        for xb, yy, tb in dl:
            xb, yy, tb = xb.to(device), yy.to(device), tb.to(device)
            opt.zero_grad()
            loss = F.huber_loss(model(xb, tb), yy, delta=1.0)
            if torch.isnan(loss): continue
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        model.eval(); vl, nv = 0.0, 0
        with torch.no_grad():
            for xb, yy, tb in vdl:
                xb, yy, tb = xb.to(device), yy.to(device), tb.to(device)
                vl += F.mse_loss(model(xb, tb), yy).item()*len(xb); nv += len(xb)
        vl /= max(nv,1); sch.step()
        if vl < best - 1e-6:
            best, best_state, pat = vl, {k: v.cpu().clone() for k,v in model.state_dict().items()}, 0
        else:
            pat += 1
            if pat >= NN_CFG['patience']: break
    if best_state: model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        pn = model(torch.FloatTensor(Xb).to(device),
                   torch.LongTensor(t_all[va_i]).to(device)).cpu().numpy()
    pred = np.zeros_like(pn)
    for tt, i in task_map.items():
        m = (t_all[va_i] == i)
        if m.any(): pred[m] = tsc[i].inverse_transform(pn[m].reshape(-1,1)).ravel()
    _oof_seed[va_i] = pred
    nn_fold_models.append((sc, tsc, model))
    log.metric(f'  NN seed {_si+1}/{len(NN_SEEDS)} fold {fold+1}/{N_FOLDS}: {time.time()-t0:.0f}s')

  _r = np.mean([r2_score(y_all[t_all==i], _oof_seed[t_all==i]) for i in task_map.values()])
  log.metric(f'  >> seed {_sd} mean OOF R2 = {_r:.4f}')
  nn_oof += _oof_seed / len(NN_SEEDS)

assert len(nn_fold_models) == len(NN_SEEDS) * N_FOLDS
nn_r2 = {tt: r2_score(y_all[t_all==i], nn_oof[t_all==i]) for tt, i in task_map.items()}
for tt in TARGET_TYPES: log.metric(f'  NN [{tt}] R2={nn_r2[tt]:.4f}')
log.metric(f'>>> NN mean OOF R2 = {np.mean(list(nn_r2.values())):.4f}  (must beat every seed above)')

[11:51:16] [  INFO] ============================================================
[11:51:16] [  INFO]   MULTI-TASK NN
[11:51:16] [  INFO] ============================================================
[11:51:17] [    OK] NN inputs row-masked
[11:53:12] [METRIC]   NN seed 1/5 fold 1/10: 115s
[11:54:24] [METRIC]   NN seed 1/5 fold 2/10: 72s
[11:55:43] [METRIC]   NN seed 1/5 fold 3/10: 79s
[11:57:32] [METRIC]   NN seed 1/5 fold 4/10: 109s
[11:58:32] [METRIC]   NN seed 1/5 fold 5/10: 59s
[11:59:51] [METRIC]   NN seed 1/5 fold 6/10: 79s
[12:01:29] [METRIC]   NN seed 1/5 fold 7/10: 98s
[12:02:45] [METRIC]   NN seed 1/5 fold 8/10: 76s
[12:03:35] [METRIC]   NN seed 1/5 fold 9/10: 50s
[12:04:34] [METRIC]   NN seed 1/5 fold 10/10: 59s
[12:04:34] [METRIC]   >> seed 42 mean OOF R2 = 0.8760
[12:06:29] [METRIC]   NN seed 2/5 fold 1/10: 115s
[12:08:23] [METRIC]   NN seed 2/5 fold 2/10: 114s
[12:09:58] [METRIC]   NN seed 2/5 fold 3/10: 95s
[12:11:43] [METRIC]   NN seed 2/5 fold 4/10: 105s
[12:13:07] [MET

## 8. SMILES 1D-CNN (reads SMILES only — no guard needed)

In [10]:
log.header('SMILES CNN')
SMILES_CHARS = list("CNOFPSIBrclnos=#()-+[]@12345678/\\.%*{}~<>^ ")
C2I = {c: i+1 for i, c in enumerate(SMILES_CHARS)}
VOCAB = len(SMILES_CHARS) + 1

def tok(s, L=CNN_CFG['max_len']):
    t = [C2I.get(c, 0) for c in s[:L]]
    return t + [0]*(L-len(t))

def aug(s, n):
    m = Chem.MolFromSmiles(s)
    if m is None: return [s]*n
    out = set()
    for _ in range(n*5):
        try: out.add(Chem.MolToSmiles(m, doRandom=True))
        except Exception: pass
        if len(out) >= n: break
    r = list(out)[:n]
    return r + [s]*(n-len(r))

class CNN(nn.Module):
    def __init__(s, p=0.3):
        super().__init__()
        s.emb = nn.Embedding(VOCAB, CNN_CFG['embed_dim'], padding_idx=0)
        s.convs = nn.ModuleList([nn.Sequential(
            nn.Conv1d(CNN_CFG['embed_dim'], CNN_CFG['n_filters'], k, padding=k//2),
            nn.BatchNorm1d(CNN_CFG['n_filters']), nn.SiLU()) for k in CNN_CFG['kernel_sizes']])
        pd_ = CNN_CFG['n_filters']*len(CNN_CFG['kernel_sizes'])*2
        s.fc = nn.Sequential(nn.Linear(pd_, CNN_CFG['fc_dim']), nn.BatchNorm1d(CNN_CFG['fc_dim']),
                             nn.SiLU(), nn.Dropout(p))
        s.heads = nn.ModuleList([nn.Sequential(nn.Linear(CNN_CFG['fc_dim'],64), nn.SiLU(),
                                               nn.Dropout(p*0.3), nn.Linear(64,1)) for _ in range(7)])
    def forward(s, x, t):
        e = s.emb(x).transpose(1,2); o = []
        for c in s.convs:
            z = c(e); o.append(z.mean(2)); o.append(z.max(2).values)
        h = s.fc(torch.cat(o, 1))
        out = torch.zeros(x.size(0), device=x.device)
        for i, hd in enumerate(s.heads):
            m = (t == i)
            if m.any(): out[m] = hd(h[m]).squeeze(-1)
        return out

class SDS(Dataset):
    def __init__(s, smi, y, t, augment=False, n=1):
        if augment and n > 1:
            X, Y, T = [], [], []
            for a, b, c in zip(smi, y, t):
                for v in aug(a, n): X.append(tok(v)); Y.append(b); T.append(c)
        else:
            X, Y, T = [tok(v) for v in smi], list(y), list(t)
        s.X = torch.LongTensor(X); s.y = torch.FloatTensor(Y); s.t = torch.LongTensor(T)
    def __len__(s): return len(s.X)
    def __getitem__(s, i): return s.X[i], s.y[i], s.t[i]

smi_all = train_df.smiles.values
cnn_oof = np.zeros(len(y_all)); cnn_fold_models = []
for fold, (tr_i, va_i) in enumerate(KFold(N_FOLDS, shuffle=True, random_state=SEED).split(smi_all)):
    t0 = time.time(); torch.manual_seed(SEED + fold)
    tsc, ya, yb = {}, y_all[tr_i].copy(), y_all[va_i].copy()
    for tt, i in task_map.items():
        ma, mb = (t_all[tr_i]==i), (t_all[va_i]==i)
        s = StandardScaler()
        if ma.any(): ya[ma] = s.fit_transform(y_all[tr_i][ma].reshape(-1,1)).ravel()
        if mb.any(): yb[mb] = s.transform(y_all[va_i][mb].reshape(-1,1)).ravel()
        tsc[i] = s
    model = CNN(CNN_CFG['dropout']).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=CNN_CFG['lr'], weight_decay=CNN_CFG['weight_decay'])
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CNN_CFG['epochs'], eta_min=1e-6)
    dl  = DataLoader(SDS(smi_all[tr_i], ya, t_all[tr_i], True, CNN_CFG['n_aug']),
                     batch_size=CNN_CFG['batch_size'], shuffle=True, drop_last=True)
    vdl = DataLoader(SDS(smi_all[va_i], yb, t_all[va_i]), batch_size=CNN_CFG['batch_size']*4)
    best, best_state, pat = 1e18, None, 0
    for ep in range(CNN_CFG['epochs']):
        model.train()
        for xb, yy, tb in dl:
            xb, yy, tb = xb.to(device), yy.to(device), tb.to(device)
            opt.zero_grad()
            loss = F.huber_loss(model(xb, tb), yy, delta=1.0)
            if torch.isnan(loss): continue
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        model.eval(); vl, nv = 0.0, 0
        with torch.no_grad():
            for xb, yy, tb in vdl:
                xb, yy, tb = xb.to(device), yy.to(device), tb.to(device)
                vl += F.mse_loss(model(xb, tb), yy).item()*len(xb); nv += len(xb)
        vl /= max(nv,1); sch.step()
        if vl < best - 1e-6:
            best, best_state, pat = vl, {k: v.cpu().clone() for k,v in model.state_dict().items()}, 0
        else:
            pat += 1
            if pat >= CNN_CFG['patience']: break
    if best_state: model.load_state_dict(best_state)
    model.eval()
    vds = SDS(smi_all[va_i], yb, t_all[va_i]); pl = []
    with torch.no_grad():
        for xb, _, tb in DataLoader(vds, batch_size=512):
            pl.append(model(xb.to(device), tb.to(device)).cpu().numpy())
    pn = np.concatenate(pl); pred = np.zeros_like(pn)
    for tt, i in task_map.items():
        m = (t_all[va_i]==i)
        if m.any(): pred[m] = tsc[i].inverse_transform(pn[m].reshape(-1,1)).ravel()
    cnn_oof[va_i] = pred; cnn_fold_models.append((tsc, model))
    log.metric(f'  CNN fold {fold+1}/{N_FOLDS}: {time.time()-t0:.0f}s')

cnn_r2 = {tt: r2_score(y_all[t_all==i], cnn_oof[t_all==i]) for tt, i in task_map.items()}
for tt in TARGET_TYPES: log.metric(f'  CNN [{tt}] R2={cnn_r2[tt]:.4f}')
log.metric(f'>>> CNN mean OOF R2 = {np.mean(list(cnn_r2.values())):.4f}')

[13:01:14] [  INFO] ============================================================
[13:01:14] [  INFO]   SMILES CNN
[13:01:14] [  INFO] ============================================================
[13:08:15] [METRIC]   CNN fold 1/10: 420s
[13:15:14] [METRIC]   CNN fold 2/10: 420s
[13:30:56] [METRIC]   CNN fold 3/10: 942s
[13:42:06] [METRIC]   CNN fold 4/10: 670s
[13:48:54] [METRIC]   CNN fold 5/10: 409s
[13:55:06] [METRIC]   CNN fold 6/10: 371s
[14:14:25] [METRIC]   CNN fold 7/10: 1159s
[14:23:56] [METRIC]   CNN fold 8/10: 571s
[14:29:11] [METRIC]   CNN fold 9/10: 316s
[14:38:32] [METRIC]   CNN fold 10/10: 560s
[14:38:32] [METRIC]   CNN [tg] R2=0.8585
[14:38:32] [METRIC]   CNN [egc] R2=0.8673
[14:38:32] [METRIC]   CNN [egb] R2=0.9182
[14:38:32] [METRIC]   CNN [eps] R2=0.7622
[14:38:32] [METRIC]   CNN [nc] R2=0.8382
[14:38:32] [METRIC]   CNN [ei] R2=0.8099
[14:38:32] [METRIC]   CNN [eea] R2=0.8888
[14:38:32] [METRIC] >>> CNN mean OOF R2 = 0.8490


## 9. Test predictions

In [11]:
log.header('TEST PREDICTIONS')
test_pred = {}
for kind in ['lgbm', 'xgb', 'cb']:
    p = np.zeros(len(test_df))
    for tt in TARGET_TYPES:
        m = (test_df.target_type == tt).values
        Xte = drop_leaky(test_features[m], tt)          # SAME guard as training
        preds = np.column_stack([mm.predict(Xte.values if kind=='cb' else Xte)
                                 for mm in tree_models[kind][tt]])
        p[m] = preds.mean(1)
    test_pred[kind] = p
    log.info(f'  {kind} done')

p = np.zeros(len(test_df))
for sc, tsc, model in nn_fold_models:                    # 5 seeds x 10 folds = 50 models
    Xs = np.nan_to_num(sc.transform(X_nn_test)).astype(np.float32)
    out = np.zeros(len(Xs))
    with torch.no_grad():
        for s0 in range(0, len(Xs), 512):
            e = min(s0+512, len(Xs))
            out[s0:e] = model(torch.FloatTensor(Xs[s0:e]).to(device),
                              torch.LongTensor(t_test[s0:e]).to(device)).cpu().numpy()
    q = np.zeros_like(out)
    for tt, i in task_map.items():
        m = (t_test == i)
        if m.any(): q[m] = tsc[i].inverse_transform(out[m].reshape(-1,1)).ravel()
    p += q / len(nn_fold_models)
test_pred['nn'] = p
log.info('  nn done')

p = np.zeros(len(test_df))
tds = SDS(test_df.smiles.values, np.zeros(len(test_df)), t_test)
for tsc, model in cnn_fold_models:
    pl = []
    with torch.no_grad():
        for xb, _, tb in DataLoader(tds, batch_size=512):
            pl.append(model(xb.to(device), tb.to(device)).cpu().numpy())
    out = np.concatenate(pl); q = np.zeros_like(out)
    for tt, i in task_map.items():
        m = (t_test == i)
        if m.any(): q[m] = tsc[i].inverse_transform(out[m].reshape(-1,1)).ravel()
    p += q / len(cnn_fold_models)
test_pred['cnn'] = p
log.ok('all base predictions generated')

[14:38:32] [  INFO] ============================================================
[14:38:32] [  INFO]   TEST PREDICTIONS
[14:38:32] [  INFO] ============================================================
[14:38:38] [  INFO]   lgbm done
[14:38:57] [  INFO]   xgb done
[14:38:59] [  INFO]   cb done
[14:39:23] [  INFO]   nn done
[14:39:26] [    OK] all base predictions generated


## 10. Ridge stacking

In [12]:
log.header('STACKING')
oof_dict = {'lgbm': tree_oof['lgbm'], 'xgb': tree_oof['xgb'], 'cb': tree_oof['cb'],
            'nn': nn_oof, 'cnn': cnn_oof}
names = sorted(oof_dict)
stack_r2, final = {}, np.zeros(len(test_df))

for tt in TARGET_TYPES:
    m  = (train_df.target_type == tt).values
    mt = (test_df.target_type == tt).values
    mX = np.nan_to_num(np.column_stack([oof_dict[n][m] for n in names]))
    my = train_df.loc[m, 'target'].values
    tX = np.nan_to_num(np.column_stack([test_pred[n][mt] for n in names]))

    best_a, best_s = 1.0, -1e18
    for a in [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]:
        sc_ = []
        for ti, vi in KFold(3, shuffle=True, random_state=SEED+200).split(mX):
            s = StandardScaler(); r = Ridge(alpha=a, random_state=SEED)
            r.fit(np.nan_to_num(s.fit_transform(mX[ti])), my[ti])
            sc_.append(r2_score(my[vi], r.predict(np.nan_to_num(s.transform(mX[vi])))))
        if np.mean(sc_) > best_s: best_s, best_a = np.mean(sc_), a

    oof_m = np.zeros(len(my))
    for ti, vi in KFold(N_FOLDS, shuffle=True, random_state=SEED).split(mX):
        s = StandardScaler(); r = Ridge(alpha=best_a, random_state=SEED)
        r.fit(np.nan_to_num(s.fit_transform(mX[ti])), my[ti])
        oof_m[vi] = r.predict(np.nan_to_num(s.transform(mX[vi])))
    stack_r2[tt] = r2_score(my, oof_m)

    s = StandardScaler(); r = Ridge(alpha=best_a, random_state=SEED)
    r.fit(np.nan_to_num(s.fit_transform(mX)), my)
    final[mt] = r.predict(np.nan_to_num(s.transform(tX)))
    log.metric(f'  [{tt}] alpha={best_a:<7g} stack OOF R2={stack_r2[tt]:.4f}')

log.metric(f'>>> STACK MEAN OOF R2 = {np.mean(list(stack_r2.values())):.4f}')

hdr = f'{"target":<8}{"lgbm":>9}{"xgb":>9}{"cb":>9}{"nn":>9}{"cnn":>9}{"stack":>9}'
print('\n' + hdr); print('-'*len(hdr))
for tt in TARGET_TYPES:
    print(f'{tt:<8}{tree_r2["lgbm"][tt]:>9.4f}{tree_r2["xgb"][tt]:>9.4f}{tree_r2["cb"][tt]:>9.4f}'
          f'{nn_r2[tt]:>9.4f}{cnn_r2[tt]:>9.4f}{stack_r2[tt]:>9.4f}')
print('-'*len(hdr))
print(f'{"MEAN":<8}' + ''.join(f'{np.mean(list(d.values())):>9.4f}' for d in
      [tree_r2['lgbm'], tree_r2['xgb'], tree_r2['cb'], nn_r2, cnn_r2, stack_r2]))

[14:39:26] [  INFO] ============================================================
[14:39:26] [  INFO]   STACKING
[14:39:26] [  INFO] ============================================================
[14:39:26] [METRIC]   [tg] alpha=0.1     stack OOF R2=0.9154
[14:39:26] [METRIC]   [egc] alpha=100     stack OOF R2=0.9170
[14:39:26] [METRIC]   [egb] alpha=1       stack OOF R2=0.9452
[14:39:26] [METRIC]   [eps] alpha=1       stack OOF R2=0.8360
[14:39:26] [METRIC]   [nc] alpha=1       stack OOF R2=0.9045
[14:39:26] [METRIC]   [ei] alpha=1       stack OOF R2=0.8841
[14:39:26] [METRIC]   [eea] alpha=1       stack OOF R2=0.9226
[14:39:26] [METRIC] >>> STACK MEAN OOF R2 = 0.9035

target       lgbm      xgb       cb       nn      cnn    stack
--------------------------------------------------------------
tg         0.9033   0.9078   0.8956   0.8993   0.8585   0.9154
egc        0.9084   0.9108   0.9031   0.8832   0.8673   0.9170
egb        0.9121   0.9103   0.9097   0.9372   0.9182   0.9452
eps      

## 11. Physics blending

Two disjoint passes. The first covers rows where the partner's **true** value is in train; the
second covers the remainder using **predicted** partners — which works because `ei = egc + eea`
is near-exact and `egc` has 2028 labels against `ei`'s 222, so routing through the identity
bypasses `ei`'s own label ceiling.

Calibration and blend weight are both fitted on train OOF and shrunk 25%; if physics does not
help a property the weight comes out 0 and that property is left untouched.

In [13]:
PHYS = {                       # target -> (source properties, direct estimator)
    'ei':  (['egc', 'eea'], lambda d: d[:, 0] + d[:, 1]),   # fundamental gap, R2=0.963
    'eea': (['ei', 'egc'],  lambda d: d[:, 0] - d[:, 1]),   # R2=0.971
    'egb': (['egc'],        lambda d: d[:, 0]),             # R2=0.892
    'eps': (['nc'],         lambda d: d[:, 0] ** 2),        # Maxwell
    'nc':  (['eps'],        lambda d: np.sqrt(np.clip(d[:, 0], 0, None))),
}
SHRINK = 0.75

def _stack_oof_for(p):
    mtr = (train_df.target_type == p).values
    y = train_df.loc[mtr, 'target'].values
    mX = np.nan_to_num(np.column_stack([oof_dict[n][mtr] for n in names]))
    o = np.zeros(len(y))
    for a, b in KFold(N_FOLDS, shuffle=True, random_state=SEED).split(mX):
        sc = StandardScaler(); r = Ridge(alpha=1.0, random_state=SEED)
        r.fit(np.nan_to_num(sc.fit_transform(mX[a])), y[a])
        o[b] = r.predict(np.nan_to_num(sc.transform(mX[b])))
    return mtr, y, o

def _true_partner(df, props):
    c = df['smiles'].map(_cmap)
    return np.column_stack([c.map(_truth[q]).values.astype(np.float64) for q in props])

def _fit_blend(est, y_sub, model_sub):
    """Out-of-fold calibration of the physics estimate + simplex weight, both on train."""
    cal = np.zeros(len(est))
    for a, b in KFold(5, shuffle=True, random_state=SEED).split(est):
        A = np.c_[est[a], np.ones(len(a))]
        w_, *_ = np.linalg.lstsq(A, y_sub[a], rcond=None)
        cal[b] = np.c_[est[b], np.ones(len(b))] @ w_
    bw, br = 0.0, -1e18
    for w in np.arange(0, 1.001, 0.05):
        r = r2_score(y_sub, (1-w)*model_sub + w*cal)
        if r > br: br, bw = r, w
    A = np.c_[est, np.ones(len(est))]
    coef, *_ = np.linalg.lstsq(A, y_sub, rcond=None)
    return cal, bw, br, coef

# ---------- pass 1: rows with TRUE partners ----------
log.header('PHYSICS BLEND -- TRUE PARTNERS')
applied_cov = 0
for p, (srcs, fn) in PHYS.items():
    mtr, y, stack_oof = _stack_oof_for(p)
    mte = (test_df.target_type == p).values
    ok  = np.isfinite(_true_partner(train_df[mtr], srcs)).all(1)
    if ok.sum() < 25:
        log.info(f'  {p}: only {ok.sum()} covered train rows -- skipped'); continue
    est = fn(_true_partner(train_df[mtr], srcs)[ok])
    cal, bw, br, coef = _fit_blend(est, y[ok], stack_oof[ok])
    w_use = SHRINK * bw
    if w_use <= 0:
        log.info(f'  {p}: physics adds nothing (w=0) -- untouched'); continue
    Dte = _true_partner(test_df[mte], srcs); okte = np.isfinite(Dte).all(1)
    if okte.sum() == 0: continue
    cal_te = np.c_[fn(Dte[okte]), np.ones(okte.sum())] @ coef
    idx = np.where(mte)[0][okte]
    final[idx] = (1-w_use)*final[idx] + w_use*cal_te
    applied_cov += len(idx)
    log.metric(f'  {p:4s} cov train {ok.sum():4d}/{mtr.sum():<4d} test {okte.sum():4d}/{mte.sum():<4d}'
               f' | stack {r2_score(y[ok], stack_oof[ok]):.4f} phys {r2_score(y[ok], cal):.4f}'
               f' blend {br:.4f} | w={bw:.2f}->{w_use:.2f}')
log.ok(f'true-partner physics applied to {applied_cov} test rows')

# ---------- pass 2: the remainder, via PREDICTED partners ----------
log.header('PHYSICS BLEND -- PREDICTED PARTNERS')
PRED_tr, PRED_te = {}, {}
for q in DFT_PROPS:
    PRED_tr[q] = np.mean([m.predict(drop_leaky(train_features, q)) for m in tree_models['lgbm'][q]], axis=0)
    PRED_te[q] = np.mean([m.predict(drop_leaky(test_features,  q)) for m in tree_models['lgbm'][q]], axis=0)
log.info('  partner predictions ready')

applied_unc = 0
for p, (srcs, fn) in PHYS.items():
    mtr, y, stack_oof = _stack_oof_for(p)
    mte = (test_df.target_type == p).values
    unc_tr = ~np.isfinite(_true_partner(train_df[mtr], srcs)).all(1)
    unc_te = ~np.isfinite(_true_partner(test_df[mte], srcs)).all(1)
    if unc_tr.sum() < 25 or unc_te.sum() == 0:
        log.info(f'  {p}: {unc_tr.sum()} uncovered train / {unc_te.sum()} test -- skipped'); continue
    est = fn(np.column_stack([PRED_tr[q][mtr] for q in srcs])[unc_tr])
    cal, bw, br, coef = _fit_blend(est, y[unc_tr], stack_oof[unc_tr])
    w_use = SHRINK * bw
    if w_use <= 0:
        log.info(f'  {p}: predicted-physics adds nothing (w=0) -- untouched'); continue
    est_te = fn(np.column_stack([PRED_te[q][mte] for q in srcs])[unc_te])
    cal_te = np.c_[est_te, np.ones(len(est_te))] @ coef
    idx = np.where(mte)[0][unc_te]
    final[idx] = (1-w_use)*final[idx] + w_use*cal_te
    applied_unc += len(idx)
    log.metric(f'  {p:4s} uncov train {unc_tr.sum():4d}/{mtr.sum():<4d} test {unc_te.sum():4d}/{mte.sum():<4d}'
               f' | stack {r2_score(y[unc_tr], stack_oof[unc_tr]):.4f} predphys {r2_score(y[unc_tr], cal):.4f}'
               f' blend {br:.4f} | w={bw:.2f}->{w_use:.2f}')
log.ok(f'predicted-partner physics applied to {applied_unc} test rows')
assert np.isfinite(final).all(), 'physics blending produced non-finite values'

[14:39:27] [  INFO] ============================================================
[14:39:27] [  INFO]   PHYSICS BLEND -- TRUE PARTNERS
[14:39:27] [  INFO] ============================================================
[14:39:27] [METRIC]   ei   cov train   59/222  test   55/148  | stack 0.9127 phys 0.9606 blend 0.9705 | w=0.70->0.53
[14:39:27] [METRIC]   eea  cov train   59/221  test   51/147  | stack 0.9511 phys 0.9723 blend 0.9761 | w=0.70->0.53
[14:39:27] [METRIC]   egb  cov train  175/337  test  124/224  | stack 0.9607 phys 0.9258 blend 0.9612 | w=0.10->0.08
[14:39:27] [METRIC]   eps  cov train  134/229  test   95/153  | stack 0.8925 phys 0.8499 blend 0.9041 | w=0.30->0.23
[14:39:27] [METRIC]   nc   cov train  134/229  test   95/153  | stack 0.9450 phys 0.8335 blend 0.9536 | w=0.20->0.15
[14:39:27] [    OK] true-partner physics applied to 420 test rows
[14:39:27] [  INFO] ============================================================
[14:39:27] [  INFO]   PHYSICS BLEND -- PREDICTED PART

## 12. Submission

In [14]:
log.header('SUBMISSION')

# clip to each property's observed range -- a negative bandgap is not a polymer
for tt in TARGET_TYPES:
    m = (test_df.target_type == tt).values
    v = train_df.loc[train_df.target_type == tt, 'target']
    lo, hi = v.min(), v.max(); pad = 0.05*(hi-lo)
    final[m] = np.clip(final[m], lo-pad, hi+pad)

bad = ~np.isfinite(final)
if bad.any():
    log.warn(f'{bad.sum()} non-finite predictions -> LightGBM fallback')
    final[bad] = test_pred['lgbm'][bad]

# ---- COMPLIANCE: prove no external/archive label ever entered the pipeline ----
_tmp2 = train_df.assign(_c=train_df.smiles.map(_cmap))
for q in DFT_PROPS:
    assert len(_truth[q]) == _tmp2[_tmp2.target_type == q]._c.nunique(), \
        f'{q}: partner table contains labels not from train.csv'
_present = glob.glob(f'{DATA_DIR}/**/archive/*', recursive=True)
log.ok(f'COMPLIANCE: only train.csv/test.csv used; archive/ present on disk '
       f'({len(_present)} files) but never read')

sub = pd.DataFrame({'id': test_df.id.values, 'target': final})
assert len(sub) == len(test_df)
assert sub.target.notna().all() and np.isfinite(sub.target.values).all()
assert sub.id.nunique() == len(sub)
sub.to_csv(os.path.join(WORK_DIR, 'submission.csv'), index=False)
log.ok(f'submission.csv written {sub.shape}')
print(sub.groupby(test_df.target_type).target.agg(['min','mean','max']).round(3))
sub.head()

[14:40:15] [  INFO] ============================================================
[14:40:15] [  INFO]   SUBMISSION
[14:40:15] [  INFO] ============================================================
[14:40:15] [    OK] COMPLIANCE: only train.csv/test.csv used; archive/ present on disk (4 files) but never read
[14:40:15] [    OK] submission.csv written (4940, 2)
                 min     mean      max
target_type                           
eea            0.156    2.322    4.460
egb            1.217    4.280    7.699
egc            0.548    4.518    8.426
ei             4.595    6.139    9.637
eps            3.144    4.622    7.819
nc             1.560    1.976    2.610
tg          -100.293  137.370  419.461


,id,target
0,1,4.147811
1,2,2.490642
2,3,309.420351
3,4,-31.226955
4,5,4.363028
